# 0 · Setup check

Run this notebook **at home, after the [preparation](../preparation/readme.md)** and once more at the start of the training day.

Use `Kernel > Restart Kernel and Run All Cells...`. Every check should print ✅. If one prints ❌, the message tells you what to fix.
If you cannot fix it yourself, bring the output to the training - we will sort it out during the setup check.

## 1. Python and virtual environment

In [1]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
in_venv = ".venv" in Path(sys.executable).parts
print("✅ Running inside the project venv" if in_venv else
      "❌ Not running from the project .venv - start JupyterLab with `uv run --env-file secrets jupyter lab` from the training folder")

Python: 3.13.16
✅ Running inside the project venv


## 2. Python packages

In [2]:
from importlib import metadata

packages = ["jupyterlab", "bia-bob", "pandas", "pandera", "fg-data-profiling", "frictionless", "requests"]
for pkg in packages:
    try:
        print(f"✅ {pkg:20s} {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"❌ {pkg:20s} missing - run `uv sync` in the training folder")

✅ jupyterlab           4.6.6
✅ bia-bob              0.37.2
✅ pandas               2.3.3
✅ pandera              0.36.0
✅ fg-data-profiling    4.20.0
✅ frictionless         5.21.0
✅ requests             2.34.2


## 3. Secrets / environment variables

The values are **not** printed, only whether they are set. Never show your API key on a shared screen.

In [3]:
import os

required = ["ENDPOINT_URL", "OPENAI_API_KEY", "LLM_MODEL", "LLM_MODEL_CODER", "SYSTEM_PROMPT_DATA_STEWARD"]
optional = ["ZENODO_SANDBOX_TOKEN"]

for var in required:
    print(("✅ " if os.getenv(var) else "❌ ") + var + (" is set" if os.getenv(var) else
          " is missing - fill it in `secrets` and restart JupyterLab with --env-file secrets"))
for var in optional:
    print(("✅ " if os.getenv(var) else "➖ ") + var + (" is set" if os.getenv(var) else " not set (optional, only for the Zenodo deposit)"))

✅ ENDPOINT_URL is set
✅ OPENAI_API_KEY is set
✅ LLM_MODEL is set
✅ LLM_MODEL_CODER is set
✅ SYSTEM_PROMPT_DATA_STEWARD is set
✅ ZENODO_SANDBOX_TOKEN is set


## 4. LLM endpoint and models

In [4]:
from bia_bob import available_models

try:
    models = available_models(endpoint=os.getenv("ENDPOINT_URL"))
    print(f"✅ Endpoint reachable, {len(models)} models available")
    for var in ["LLM_MODEL", "LLM_MODEL_CODER"]:
        name = os.getenv(var)
        print(("✅ " if name in models else "❌ ") + f"{var} = {name!r}" +
              ("" if name in models else " is not offered by this endpoint - pick one from the list below"))
    if not all(os.getenv(v) in models for v in ["LLM_MODEL", "LLM_MODEL_CODER"]):
        display(models)
except Exception as e:
    print("❌ Could not list models:", type(e).__name__, e)

✅ Endpoint reachable, 75 models available
✅ LLM_MODEL = 'alias-fast'
✅ LLM_MODEL_CODER = 'alias-code'


## 5. First prompt

In [5]:
from bia_bob import bob

bob.initialize(
    endpoint=os.getenv("ENDPOINT_URL"),
    model=os.getenv("LLM_MODEL"),
    system_prompt=os.getenv("SYSTEM_PROMPT_DATA_STEWARD"),
)

In [6]:
%bob Reply with two sentences confirming who you are, what you can support with, and that you are ready. No code.

I am an AI assistant specialized in supporting Data Stewards with research data management, FAIR practices, metadata, governance, compliance, repository selection, and related stewardship tasks. I’m ready to help you with planning, documentation, quality checks, and any other data stewardship needs you have.

If the last cell printed an answer from the assistant: **you are ready for the training.**